Add Dataset and Verify Integrity. 

Remove any null values.

Remove any duplicates.

In [2]:
import pandas as pd
import numpy as np

# Load Dataset
df = pd.read_excel(r'C:\Users\Mindy\Desktop\DataDrills\Insurance.xlsx')

# Audit and Clean Data
print("--- Dataset Information---")
print(df.info())
print("\n---Null Count ---")
print(df.isnull().sum())
print("\n--- Duplicate Records ---")
print(df.duplicated().sum())

--- Dataset Information---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB
None

---Null Count ---
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

--- Duplicate Records ---
1


One duplicate member row was found.

Remove the duplicated member.

In [3]:
duplicate_count = df.duplicated().sum()
if duplicate_count > 0:
        df = df.drop_duplicates(keep='first')

# Verify
print("--- Duplicate Records ---")
print(df.duplicated().sum())

--- Duplicate Records ---
0


Create a Risk Tier for members based upon smoking habits and BMI.

In [4]:
conditions = [
    (df['smoker'] == 'yes') & (df['bmi'] >=30), #critical risk
    (df['smoker'] == 'yes') | (df['bmi'] >=30)  #moderate risk
]
choices = ['Critical Risk', 'Moderate Risk']

# ELSE low risk
df['Risk_Tier'] = np.select(conditions, choices, default='Low Risk')

# Verify
print(df['Risk_Tier'].value_counts())

Risk_Tier
Moderate Risk    690
Low Risk         502
Critical Risk    145
Name: count, dtype: int64


Financial Metric Aggregations

In [5]:
# Cost-containment financial metric by risk tier

cost_summary = df.groupby('Risk_Tier').agg(
    Total_Members=('charges', 'count'),
    Average_Claim_Cost=('charges', 'mean'),
    Total_Plan_Spend=('charges', 'sum')
).round(2).reset_index()

print("--- TPA Health Plan Cost Breakdown ---")
print(cost_summary)

--- TPA Health Plan Cost Breakdown ---
       Risk_Tier  Total_Members  Average_Claim_Cost  Total_Plan_Spend
0  Critical Risk            145            41557.99        6025908.53
1       Low Risk            502             7977.03        4004468.82
2  Moderate Risk            690            11193.92        7723808.08


Export Cleaned and Updated Data to Power BI

In [6]:
df.to_csv('clean_insurance_data.csv', index=False)
print("Success")

Success
